## PHÂN TÍCH DỮ LIỆU DU LỊCH VIỆT NAM

### Nội dung

- **3.1 Donut Chart** – Cơ cấu khách trong nước và quốc tế năm 2024
- **3.2 Stacked Bar Chart** – Thay đổi cơ cấu khách giai đoạn 2015–2024
- **3.3 Line Chart** – Xu hướng doanh thu du lịch
- **3.4 Grouped Bar Chart** – So sánh lượng khách lưu trú và lữ hành
- **3.5 Area Chart** – Xu hướng tổng lượng khách du lịch
- **3.6 Horizontal Bar Chart** – Top 10 tỉnh/thành theo doanh thu năm 2023
- **3.7 Column Chart** – Mức tăng/giảm tổng doanh thu theo năm
- **3.8 Treemap** – Cơ cấu doanh thu du lịch lữ hành theo vùng năm 2023
- **3.9 Correlation Heatmap** – Tương quan giữa các chỉ tiêu du lịch
- **3.10 Tổng hợp các insight chính**

In [18]:
import sys
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Hiển thị biểu đồ trực tiếp trong VS Code Notebook
pio.renderers.default = "plotly_mimetype"

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


# Hàm định dạng số theo kiểu Việt Nam
def fmt_vi(value, decimals=0):
    s = f"{value:,.{decimals}f}"
    return s.replace(",", "_").replace(".", ",").replace("_", ".")


# Xác định thư mục gốc project
ROOT = Path(sys.prefix).parent

DATA_PATH = ROOT / "data" / "processed"
FIGURES = ROOT / "outputs" / "figures"

FIGURES.mkdir(parents=True, exist_ok=True)

file_nam = DATA_PATH / "du_lieu_du_lich_nam_clean.csv"
file_dp = DATA_PATH / "du_lieu_du_lich_dia_phuong_clean.csv"

df_nam = pd.read_csv(file_nam, encoding="utf-8-sig")
df_dia_phuong = pd.read_csv(file_dp, encoding="utf-8-sig")

print("Dữ liệu theo năm:", df_nam.shape)
print("Dữ liệu địa phương:", df_dia_phuong.shape)
print("Giai đoạn:", df_nam["Năm"].min(), "-", df_nam["Năm"].max())


Dữ liệu theo năm: (10, 13)
Dữ liệu địa phương: (700, 5)
Giai đoạn: 2015 - 2024


In [20]:
# =========================================================
# KIỂM TRA NHANH DỮ LIỆU ĐẦU VÀO
# =========================================================

missing_nam = df_nam.isna().sum()
missing_nam = missing_nam[missing_nam > 0]

print("CÁC CỘT CÒN GIÁ TRỊ THIẾU")
display(missing_nam.to_frame("Số giá trị thiếu"))

print("\nCẤP DỮ LIỆU ĐỊA PHƯƠNG")
display(
    df_dia_phuong["Cấp dữ liệu"]
    .value_counts()
    .to_frame("Số quan sát")
)


CÁC CỘT CÒN GIÁ TRỊ THIẾU


,Số giá trị thiếu
Khách nghỉ qua đêm (Nghìn lượt),6
Khách trong ngày (Nghìn lượt),6



CẤP DỮ LIỆU ĐỊA PHƯƠNG


,Số quan sát
Cấp dữ liệu,
Tỉnh/thành,630
Vùng,60
Cả nước,10


### Nhận xét dữ liệu đầu vào

- Hai bộ dữ liệu đã được làm sạch và chuẩn bị trước khi thực hiện trực quan hóa.
- Hai biến `Khách nghỉ qua đêm` và `Khách trong ngày` còn giá trị thiếu nên không được sử dụng trong 9 biểu đồ của Chương 3.
- Khi phân tích địa phương, dữ liệu được lọc theo đúng cấp `Tỉnh/thành` hoặc `Vùng` để tránh trùng với số liệu tổng hợp ở cấp `Cả nước`.
- Số liệu năm 2024 trong bộ dữ liệu là **số liệu sơ bộ**.

## 3.1. Donut Chart – Cơ cấu khách trong nước và quốc tế năm 2024

**Mục tiêu:** Phân tích tỷ trọng khách trong nước và khách quốc tế tại các cơ sở lưu trú trong năm 2024.

In [31]:
# =========================================================
# BIỂU ĐỒ 1 - DONUT CHART
# =========================================================

domestic_col = "Khách trong nước - cơ sở lưu trú (Nghìn lượt)"
international_col = "Khách quốc tế - cơ sở lưu trú (Nghìn lượt)"

row_2024 = df_nam[df_nam["Năm"] == 2024].iloc[0]

pie_data = pd.DataFrame({
    "Loại khách": [
        "Khách trong nước",
        "Khách quốc tế"
    ],
    "Lượng khách": [
        row_2024[domestic_col],
        row_2024[international_col]
    ]
})

pie_data["Tỷ trọng"] = (
    pie_data["Lượng khách"]
    / pie_data["Lượng khách"].sum()
    * 100
)

pie_data["Nhãn"] = pie_data.apply(
    lambda r:
    f"{r['Loại khách']}<br>"
    f"{fmt_vi(r['Lượng khách'], 0)} nghìn lượt<br>"
    f"{fmt_vi(r['Tỷ trọng'], 1)}%",
    axis=1
)

fig1 = px.pie(
    pie_data,
    names="Loại khách",
    values="Lượng khách",
    hole=0.45,
    title="Cơ cấu khách trong nước và quốc tế tại cơ sở lưu trú năm 2024"
)

fig1.update_traces(
    text=pie_data["Nhãn"],
    textinfo="text",
    textposition="inside",
    textfont_size=15,
    marker=dict(
        line=dict(width=2)
    )
)

fig1.update_layout(
    template="plotly_white",
    showlegend=False,
    width=900,
    height=600,
    annotations=[
        dict(
            text="<b>2024</b><br>Sơ bộ",
            x=0.5,
            y=0.5,
            showarrow=False,
            font_size=16
        )
    ]
)

file_fig1 = FIGURES / "01_co_cau_khach_2024.png"

fig1.write_image(file_fig1, scale=2)

fig1.show()


### Nhận xét Biểu đồ 1

- Năm 2024, **khách trong nước đạt khoảng 192.273 nghìn lượt, chiếm 76,3%** tổng lượng khách lưu trú.
- **Khách quốc tế đạt khoảng 59.624 nghìn lượt, chiếm 23,7%**.
- Kết quả cho thấy khách trong nước vẫn giữ vai trò chủ đạo trong cơ cấu khách tại các cơ sở lưu trú.

## 3.2. Stacked Bar Chart – Thay đổi cơ cấu khách giai đoạn 2015–2024

**Mục tiêu:** Theo dõi sự thay đổi tỷ trọng khách trong nước và khách quốc tế tại cơ sở lưu trú qua từng năm.

In [ ]:
# =========================================================
# BIỂU ĐỒ 2 - STACKED BAR
# =========================================================

share_df = df_nam[
    ["Năm", domestic_col, international_col]
].copy()

share_df["Tổng khách"] = (
    share_df[domestic_col]
    + share_df[international_col]
)

share_df["Khách trong nước (%)"] = (
    share_df[domestic_col]
    / share_df["Tổng khách"]
    * 100
)

share_df["Khách quốc tế (%)"] = (
    share_df[international_col]
    / share_df["Tổng khách"]
    * 100
)

fig2 = go.Figure()

fig2.add_trace(
    go.Bar(
        x=share_df["Năm"],
        y=share_df["Khách trong nước (%)"],
        name="Khách trong nước",
        text=[
            f"{fmt_vi(v, 1)}%"
            for v in share_df["Khách trong nước (%)"]
        ],
        textposition="inside"
    )
)

fig2.add_trace(
    go.Bar(
        x=share_df["Năm"],
        y=share_df["Khách quốc tế (%)"],
        name="Khách quốc tế",
        text=[
            f"{fmt_vi(v, 1)}%"
            for v in share_df["Khách quốc tế (%)"]
        ],
        textposition="inside"
    )
)

fig2.update_layout(
    title="Thay đổi cơ cấu khách trong nước và quốc tế giai đoạn 2015–2024",
    xaxis_title="Năm",
    yaxis_title="Tỷ trọng (%)",
    barmode="stack",
    template="plotly_white",
    legend_title="Loại khách",
    width=1100,
    height=650,
    yaxis=dict(
        range=[0, 100],
        ticksuffix="%"
    )
)

fig2.update_xaxes(dtick=1)

file_fig2 = FIGURES / "02_co_cau_khach.png"

fig2.write_image(file_fig2, scale=2)

fig2.show()


### Nhận xét Biểu đồ 2

- Khách trong nước luôn chiếm tỷ trọng lớn trong toàn bộ giai đoạn 2015–2024.
- Tỷ trọng khách quốc tế giảm mạnh trong giai đoạn 2020–2021, xuống khoảng **5,0% vào năm 2021**.
- Từ năm 2022, khách quốc tế phục hồi rõ rệt, đạt **18,8% năm 2023** và **23,7% năm 2024**.
- Cơ cấu khách cho thấy thị trường quốc tế đang dần phục hồi sau giai đoạn suy giảm.

## 3.3. Line Chart – Xu hướng doanh thu du lịch giai đoạn 2015–2024

**Mục tiêu:** Phân tích sự thay đổi của doanh thu cơ sở lưu trú và doanh thu cơ sở lữ hành theo thời gian.

In [33]:
# =========================================================
# BIỂU ĐỒ 3 - LINE CHART DOANH THU
# =========================================================

rev_stay = "Doanh thu cơ sở lưu trú (Tỷ đồng)"
rev_tour = "Doanh thu cơ sở lữ hành (Tỷ đồng)"

fig3 = go.Figure()

fig3.add_trace(
    go.Scatter(
        x=df_nam["Năm"],
        y=df_nam[rev_stay],
        mode="lines+markers+text",
        name="Doanh thu lưu trú",
        text=[
            fmt_vi(v, 0)
            for v in df_nam[rev_stay]
        ],
        textposition="top center",
        line=dict(width=3),
        marker=dict(size=9)
    )
)

fig3.add_trace(
    go.Scatter(
        x=df_nam["Năm"],
        y=df_nam[rev_tour],
        mode="lines+markers+text",
        name="Doanh thu lữ hành",
        text=[
            fmt_vi(v, 0)
            for v in df_nam[rev_tour]
        ],
        textposition="bottom center",
        line=dict(width=3),
        marker=dict(size=9)
    )
)

fig3.update_layout(
    title="Xu hướng doanh thu du lịch Việt Nam giai đoạn 2015–2024",
    xaxis_title="Năm",
    yaxis_title="Doanh thu (Tỷ đồng)",
    template="plotly_white",
    legend_title="Loại doanh thu",
    width=1150,
    height=650
)

fig3.update_xaxes(dtick=1)
fig3.update_yaxes(rangemode="tozero")

file_fig3 = FIGURES / "03xu_huong_doanh_thu.png"

fig3.write_image(file_fig3, scale=2)

fig3.show()

### Nhận xét Biểu đồ 3

- Doanh thu lưu trú và lữ hành tăng tương đối ổn định trong giai đoạn **2015–2019**.
- Giai đoạn **2020–2021** ghi nhận sự suy giảm mạnh; năm 2021 doanh thu lưu trú còn khoảng **23.690 tỷ đồng** và doanh thu lữ hành khoảng **8.999 tỷ đồng**.
- So với năm 2019, doanh thu lữ hành năm 2021 giảm khoảng **80%**, mạnh hơn mức giảm khoảng **65%** của doanh thu lưu trú.
- Từ năm 2022, doanh thu phục hồi nhanh và năm 2024 đạt mức cao nhất trong giai đoạn nghiên cứu.

## 3.4. Grouped Bar Chart – So sánh lượng khách lưu trú và lữ hành

**Mục tiêu:** So sánh lượng khách do cơ sở lưu trú và cơ sở lữ hành phục vụ qua từng năm trong giai đoạn 2015–2024.

In [34]:
# =========================================================
# BIỂU ĐỒ 4 - GROUPED BAR
# =========================================================

guest_stay = "Khách cơ sở lưu trú phục vụ (Nghìn lượt)"
guest_tour = "Khách cơ sở lữ hành phục vụ (Nghìn lượt)"

fig4 = go.Figure()

fig4.add_trace(
    go.Bar(
        x=df_nam["Năm"],
        y=df_nam[guest_stay],
        name="Khách lưu trú",
        text=[
            fmt_vi(v, 0)
            for v in df_nam[guest_stay]
        ],
        textposition="outside"
    )
)

fig4.add_trace(
    go.Bar(
        x=df_nam["Năm"],
        y=df_nam[guest_tour],
        name="Khách lữ hành",
        text=[
            fmt_vi(v, 0)
            for v in df_nam[guest_tour]
        ],
        textposition="outside"
    )
)

fig4.update_layout(
    title="So sánh lượng khách lưu trú và lữ hành giai đoạn 2015–2024",
    xaxis_title="Năm",
    yaxis_title="Lượng khách (Nghìn lượt)",
    barmode="group",
    template="plotly_white",
    legend_title="Loại khách",
    width=1200,
    height=680
)

fig4.update_xaxes(dtick=1)
fig4.update_yaxes(rangemode="tozero")

file_fig4 = FIGURES / "04_luong_khach.png"

fig4.write_image(file_fig4, scale=2)

fig4.show()

### Nhận xét Biểu đồ 4

- Lượng khách lưu trú luôn cao hơn đáng kể so với lượng khách lữ hành trong toàn bộ giai đoạn.
- Cả hai nhóm tăng đến năm 2019, giảm mạnh trong các năm **2020–2021** và bắt đầu phục hồi từ năm 2022.
- Năm 2021 là mức thấp nhất với khoảng **63.603 nghìn lượt khách lưu trú** và **3.565 nghìn lượt khách lữ hành**.
- Đến năm 2024, hai chỉ tiêu lần lượt đạt khoảng **251.897** và **34.371 nghìn lượt**, cao nhất trong giai đoạn.

## 3.5. Area Chart – Xu hướng tổng lượng khách du lịch

**Mục tiêu:** Thể hiện sự thay đổi về quy mô tổng lượng khách được phục vụ bởi cơ sở lưu trú và cơ sở lữ hành trong giai đoạn 2015–2024.

In [35]:
# =========================================================
# BIỂU ĐỒ 5 - AREA CHART
# Tổng lượng khách du lịch theo năm
# =========================================================

area_df = (
    df_nam[
        ["Năm", guest_stay, guest_tour]
    ]
    .sort_values("Năm", ascending=True)
    .reset_index(drop=True)
)

# Tổng lượng khách
area_df["Tổng lượng khách"] = (
    area_df[guest_stay]
    + area_df[guest_tour]
)

# Nhãn hiển thị trực tiếp
area_df["Nhãn"] = area_df["Tổng lượng khách"].apply(
    lambda x: fmt_vi(x, 0)
)

fig5 = go.Figure()

fig5.add_trace(
    go.Scatter(
        x=area_df["Năm"],
        y=area_df["Tổng lượng khách"],
        mode="lines+markers+text",
        fill="tozeroy",
        name="Tổng lượng khách",
        text=area_df["Nhãn"],
        textposition="top center",
        line=dict(width=3),
        marker=dict(size=9)
    )
)

fig5.update_layout(
    title="Xu hướng tổng lượng khách du lịch Việt Nam giai đoạn 2015–2024",
    xaxis_title="Năm",
    yaxis_title="Tổng lượng khách (Nghìn lượt)",
    template="plotly_white",
    width=1150,
    height=650,
    showlegend=False,
    margin=dict(
        t=90,
        b=70,
        l=90,
        r=60
    )
)

fig5.update_xaxes(
    dtick=1
)

fig5.update_yaxes(
    rangemode="tozero"
)

file_fig5 = (
    FIGURES
    / "05_tong_luong_khach.png"
)

fig5.write_image(
    file_fig5,
    scale=2
)

fig5.show()


### Nhận xét Biểu đồ 5

- Tổng lượng khách tăng từ khoảng **126.613 nghìn lượt năm 2015** lên khoảng **197.732 nghìn lượt năm 2019**.
- Sau đó, lượng khách giảm mạnh và chạm mức thấp nhất vào năm 2021 với khoảng **67.168 nghìn lượt**.
- Từ năm 2022, tổng lượng khách phục hồi nhanh và đến năm 2024 đạt khoảng **286.268 nghìn lượt**, cao nhất trong toàn bộ giai đoạn.

## 3.6. Horizontal Bar Chart – Top 10 tỉnh/thành theo doanh thu du lịch lữ hành năm 2023

**Mục tiêu:** Xác định và so sánh các tỉnh/thành có doanh thu du lịch lữ hành cao nhất trong năm 2023.

> Năm 2023 được lựa chọn vì đây là năm gần nhất có số liệu chính thức; số liệu năm 2024 là sơ bộ.

In [36]:
# =========================================================
# BIỂU ĐỒ 6 - TOP 10 TỈNH/THÀNH
# =========================================================

revenue_local = "Doanh thu du lịch lữ hành (Tỷ đồng)"

top10_2023 = (
    df_dia_phuong[
        (df_dia_phuong["Cấp dữ liệu"] == "Tỉnh/thành") &
        (df_dia_phuong["Năm"] == 2023)
    ]
    .dropna(subset=[revenue_local])
    .nlargest(10, revenue_local)
    .sort_values(revenue_local)
    .copy()
)

top10_2023["Nhãn"] = top10_2023.apply(
    lambda r:
    f"{r['Địa phương']}: {fmt_vi(r[revenue_local], 0)}",
    axis=1
)

fig6 = go.Figure(
    go.Bar(
        x=top10_2023[revenue_local],
        y=top10_2023["Địa phương"],
        orientation="h",
        text=top10_2023["Nhãn"],
        textposition="outside"
    )
)

fig6.update_layout(
    title="Top 10 tỉnh/thành theo doanh thu du lịch lữ hành năm 2023",
    xaxis_title="Doanh thu du lịch lữ hành (Tỷ đồng)",
    yaxis_title="Tỉnh/thành",
    template="plotly_white",
    width=1100,
    height=700,
    margin=dict(
        l=140,
        r=180,
        t=90,
        b=70
    )
)

fig6.update_xaxes(rangemode="tozero")

file_fig6 = FIGURES / "06_top10_dia_phuong_2023.png"

fig6.write_image(file_fig6, scale=2)

fig6.show()


### Nhận xét Biểu đồ 6

- **TP. Hồ Chí Minh** đứng đầu với khoảng **25.580 tỷ đồng**, tiếp theo là **Hà Nội** với khoảng **20.687 tỷ đồng**.
- **Đà Nẵng** đứng thứ ba với khoảng **4.579 tỷ đồng**, cho thấy khoảng cách rất lớn giữa hai địa phương dẫn đầu và nhóm còn lại.
- TP. Hồ Chí Minh và Hà Nội chiếm khoảng **79,1% tổng doanh thu của nhóm Top 10 năm 2023**.
- Kết quả cho thấy doanh thu du lịch lữ hành có mức độ tập trung cao tại một số trung tâm du lịch lớn.

## 3.7. Column Chart – Mức tăng/giảm tổng doanh thu du lịch theo năm

**Mục tiêu:** So sánh mức thay đổi của tổng doanh thu lưu trú và lữ hành so với năm liền trước, qua đó xác định các giai đoạn tăng trưởng và suy giảm mạnh.

In [37]:
# =========================================================
# BIỂU ĐỒ 7 - COLUMN CHART
# Mức tăng/giảm tổng doanh thu theo năm
# =========================================================

change_df = (
    df_nam[
        ["Năm", rev_stay, rev_tour]
    ]
    .sort_values("Năm", ascending=True)
    .reset_index(drop=True)
)

# Tổng doanh thu
change_df["Tổng doanh thu"] = (
    change_df[rev_stay]
    + change_df[rev_tour]
)

# Mức thay đổi so với năm trước
change_df["Thay đổi"] = (
    change_df["Tổng doanh thu"]
    .diff()
)

# Bỏ năm 2015 vì không có năm trước để so sánh
change_plot = (
    change_df
    .dropna(subset=["Thay đổi"])
    .copy()
)

# Nhãn số liệu
change_plot["Nhãn"] = change_plot["Thay đổi"].apply(
    lambda x:
    f"+{fmt_vi(x, 0)}"
    if x >= 0
    else fmt_vi(x, 0)
)

# Màu phân biệt tăng và giảm
bar_colors = [
    "#2E8B57" if x >= 0 else "#C94C4C"
    for x in change_plot["Thay đổi"]
]

fig7 = go.Figure()

fig7.add_trace(
    go.Bar(
        x=change_plot["Năm"],
        y=change_plot["Thay đổi"],
        text=change_plot["Nhãn"],
        textposition="outside",
        marker_color=bar_colors,
        name="Thay đổi doanh thu"
    )
)

# Đường mốc 0
fig7.add_hline(
    y=0,
    line_width=1.5,
    line_color="gray"
)

fig7.update_layout(
    title="Mức tăng/giảm tổng doanh thu du lịch so với năm trước",
    xaxis_title="Năm",
    yaxis_title="Mức thay đổi doanh thu (Tỷ đồng)",
    template="plotly_white",
    width=1150,
    height=680,
    showlegend=False,
    margin=dict(
        t=90,
        b=70,
        l=100,
        r=60
    )
)

fig7.update_xaxes(
    dtick=1
)

file_fig7 = (
    FIGURES
    / "07_thay_doi_doanh_thu.png"
)

fig7.write_image(
    file_fig7,
    scale=2
)

fig7.show()

### Nhận xét Biểu đồ 7

- Tổng doanh thu duy trì mức tăng dương trong giai đoạn trước năm 2020.
- Năm 2020 ghi nhận mức giảm mạnh nhất, khoảng **56.593 tỷ đồng** so với năm trước; năm 2021 tiếp tục giảm khoảng **22.407 tỷ đồng**.
- Năm 2022 đánh dấu sự phục hồi mạnh nhất với mức tăng khoảng **68.513 tỷ đồng**.
- Các năm 2023 và 2024 tiếp tục tăng lần lượt khoảng **39.755 tỷ đồng** và **32.135 tỷ đồng**, cho thấy quá trình phục hồi được duy trì.

## 3.8. Treemap – Cơ cấu doanh thu du lịch lữ hành theo vùng năm 2023

**Mục tiêu:** Thể hiện quy mô và tỷ trọng doanh thu du lịch lữ hành giữa các vùng của Việt Nam trong năm 2023.

> Để đảm bảo khả năng đọc trực tiếp trên biểu đồ, bốn vùng có doanh thu cao nhất được thể hiện riêng; hai vùng có doanh thu thấp nhất được gộp vào nhóm **“Khác”**.

In [43]:
# =========================================================
# BIỂU ĐỒ 8 - TREEMAP
# Giữ 4 vùng lớn nhất, gộp 2 vùng nhỏ nhất thành "Khác"
# =========================================================

revenue_local = "Doanh thu du lịch lữ hành (Tỷ đồng)"

# Lọc dữ liệu vùng năm 2023
region_2023 = (
    df_dia_phuong[
        (df_dia_phuong["Cấp dữ liệu"] == "Vùng") &
        (df_dia_phuong["Năm"] == 2023)
    ]
    .dropna(subset=[revenue_local])
    .copy()
)

# Sắp xếp giảm dần
region_2023 = (
    region_2023
    .sort_values(revenue_local, ascending=False)
    .reset_index(drop=True)
)

# Giữ 4 vùng lớn nhất
top4 = region_2023.head(4).copy()

# Gộp các vùng còn lại thành "Khác"
others = region_2023.iloc[4:].copy()

other_row = pd.DataFrame({
    "Địa phương": ["Khác"],
    revenue_local: [others[revenue_local].sum()]
})

# Gộp lại
treemap_df = pd.concat(
    [top4[["Địa phương", revenue_local]], other_row],
    ignore_index=True
)

# Tính tỷ trọng
total_value = treemap_df[revenue_local].sum()
treemap_df["Tỷ trọng (%)"] = treemap_df[revenue_local] / total_value * 100

# Tạo nhãn hiển thị trực tiếp
treemap_df["Nhãn"] = treemap_df.apply(
    lambda r:
    f"<b>{r['Địa phương']}</b><br>"
    f"{fmt_vi(r[revenue_local], 0)} tỷ đồng<br>"
    f"{fmt_vi(r['Tỷ trọng (%)'], 1)}%",
    axis=1
)

# Vẽ biểu đồ
fig8 = go.Figure(
    go.Treemap(
        labels=treemap_df["Địa phương"],
        parents=[""] * len(treemap_df),
        values=treemap_df[revenue_local],
        text=treemap_df["Nhãn"],
        textinfo="text",
        textfont=dict(size=17),
        marker=dict(line=dict(width=2))
    )
)

fig8.update_layout(
    title="Cơ cấu doanh thu du lịch lữ hành theo vùng năm 2023",
    template="plotly_white",
    width=1100,
    height=700,
    margin=dict(t=80, l=25, r=25, b=25)
)

file_fig8 = FIGURES / "08_vung_2023_gop_khac.png"
fig8.write_image(file_fig8, scale=2)

fig8.show()

print(f"Đã lưu: {file_fig8.resolve()}")
display(
    treemap_df.style.format({
        revenue_local: "{:,.2f}",
        "Tỷ trọng (%)": "{:.1f}%"
    })
)

Đã lưu: D:\DEEP\TQH-DU-LIEU-NHOM6\outputs\figures\08_vung_2023_gop_khac.png


,Địa phương,Doanh thu du lịch lữ hành (Tỷ đồng),Tỷ trọng (%),Nhãn
0,Đông Nam Bộ,"26,502.84",41.4%,"Đông Nam Bộ26.503 tỷ đồng41,4%"
1,Đồng bằng sông Hồng,"23,864.72",37.3%,"Đồng bằng sông Hồng23.865 tỷ đồng37,3%"
2,Bắc Trung Bộ và Duyên hải miền Trung,"10,241.71",16.0%,"Bắc Trung Bộ và Duyên hải miền Trung10.242 tỷ đồng16,0%"
3,Đồng bằng sông Cửu Long,"2,328.85",3.6%,"Đồng bằng sông Cửu Long2.329 tỷ đồng3,6%"
4,Khác,"1,097.69",1.7%,"Khác1.098 tỷ đồng1,7%"


### Nhận xét Biểu đồ 8

- **Đông Nam Bộ** có doanh thu cao nhất với khoảng **26.503 tỷ đồng**, chiếm **41,4%** tổng doanh thu các vùng.
- **Đồng bằng sông Hồng** đứng thứ hai với khoảng **23.865 tỷ đồng**, chiếm **37,3%**.
- **Bắc Trung Bộ và Duyên hải miền Trung** đạt khoảng **10.242 tỷ đồng**, tương ứng **16,0%**.
- Hai vùng dẫn đầu là Đông Nam Bộ và Đồng bằng sông Hồng chiếm tổng cộng khoảng **78,7%**, cho thấy doanh thu lữ hành tập trung mạnh tại hai khu vực này.
- Nhóm **“Khác”** gồm hai vùng có doanh thu thấp nhất, được gộp lại để biểu đồ cân đối và dễ đọc hơn.

## 3.9. Correlation Heatmap – Tương quan giữa các chỉ tiêu du lịch

**Mục tiêu:** Phân tích mức độ tương quan giữa các chỉ tiêu doanh thu và lượng khách du lịch trong giai đoạn nghiên cứu.

In [38]:
# =========================================================
# BIỂU ĐỒ 9 - CORRELATION HEATMAP
# =========================================================

corr_cols = {
    rev_stay: "DT lưu trú",
    rev_tour: "DT lữ hành",
    guest_stay: "Khách lưu trú",
    domestic_col: "Khách nội địa",
    international_col: "Khách quốc tế",
    guest_tour: "Khách lữ hành"
}

corr_data = df_nam[
    list(corr_cols.keys())
].rename(
    columns=corr_cols
)

corr_matrix = corr_data.corr(
    method="pearson"
)

fig9 = px.imshow(
    corr_matrix,
    text_auto=".2f",
    zmin=-1,
    zmax=1,
    color_continuous_scale="RdBu_r",
    aspect="auto",
    title="Ma trận tương quan giữa các chỉ tiêu du lịch"
)

fig9.update_layout(
    width=950,
    height=720,
    xaxis_title="Chỉ tiêu",
    yaxis_title="Chỉ tiêu"
)

file_fig9 = FIGURES / "09_tuong_quan.png"

fig9.write_image(file_fig9, scale=2)

fig9.show()

### Nhận xét Biểu đồ 9

- Các chỉ tiêu được lựa chọn đều có **mối tương quan dương mạnh**.
- Mối tương quan cao nhất xuất hiện giữa **doanh thu lưu trú và khách lưu trú**, với hệ số **r = 0,994**.
- Doanh thu lữ hành và khách lưu trú cũng có mức tương quan rất cao, khoảng **r = 0,991**.
- Cặp có hệ số thấp nhất là **khách nội địa và khách quốc tế**, với **r = 0,816**, nhưng vẫn thể hiện mối tương quan dương mạnh.
- Kết quả cho thấy doanh thu và lượng khách có xu hướng biến động khá đồng nhất trong giai đoạn nghiên cứu.

## 3.10. Tổng hợp các insight chính

1. **Khách trong nước vẫn giữ vai trò chủ đạo**, chiếm khoảng **76,3%** tổng lượng khách tại cơ sở lưu trú năm 2024; khách quốc tế chiếm khoảng **23,7%**.

2. **Thị trường khách quốc tế phục hồi rõ rệt sau giai đoạn suy giảm.** Tỷ trọng khách quốc tế giảm xuống khoảng **5,0% năm 2021**, sau đó tăng lên **18,8% năm 2023** và **23,7% năm 2024**.

3. Du lịch Việt Nam thể hiện ba giai đoạn rõ rệt: **tăng trưởng trong 2015–2019, suy giảm mạnh trong 2020–2021 và phục hồi từ năm 2022**.

4. **Năm 2021 là giai đoạn thấp nhất về lượng khách**, với tổng lượng khách lưu trú và lữ hành khoảng **67.168 nghìn lượt**; đến năm 2024 con số này tăng lên khoảng **286.268 nghìn lượt**.

5. **Năm 2022 ghi nhận mức phục hồi doanh thu mạnh nhất**, khi tổng doanh thu tăng khoảng **68.513 tỷ đồng** so với năm 2021.

6. Doanh thu du lịch có mức độ tập trung cao. **TP. Hồ Chí Minh và Hà Nội chiếm khoảng 79,1% doanh thu của nhóm Top 10 tỉnh/thành năm 2023**; đồng thời **Đông Nam Bộ và Đồng bằng sông Hồng chiếm khoảng 78,7% tổng doanh thu lữ hành theo vùng**.

7. Các chỉ tiêu doanh thu và lượng khách có mối liên hệ chặt chẽ. Tương quan giữa **doanh thu lưu trú và khách lưu trú đạt r = 0,994**, trong khi hệ số thấp nhất trong nhóm phân tích vẫn đạt khoảng **r = 0,816**.

## Kết luận

Thông qua 9 dạng biểu đồ khác nhau, Chương 3 đã làm rõ các khía cạnh chính của dữ liệu du lịch Việt Nam gồm **cơ cấu khách, xu hướng doanh thu, quy mô lượng khách, mức biến động theo thời gian, sự khác biệt giữa các địa phương và mối tương quan giữa các chỉ tiêu**.

Kết quả cho thấy ngành du lịch tăng trưởng trong giai đoạn 2015–2019, suy giảm mạnh trong 2020–2021 và phục hồi rõ rệt từ năm 2022. Khách trong nước vẫn giữ vai trò chủ đạo, trong khi khách quốc tế đang có xu hướng phục hồi. Bên cạnh đó, doanh thu du lịch có mức độ tập trung cao tại một số địa phương và vùng kinh tế lớn.

Việc sử dụng nhiều dạng biểu đồ và hiển thị trực tiếp số liệu trên hình giúp kết quả phân tích dễ quan sát, thuận tiện cho cả báo cáo và thuyết trình.